# Deploy Hotel Booking Agent to Production with Amazon Bedrock AgentCore

In this notebook you will deploy a **production-ready hotel booking agent** to AWS using Amazon Bedrock AgentCore — step by step.

## What You Will Build

| Component | AWS Service | Purpose |
|-----------|------------|----------|
| Data layer | DynamoDB (3 tables) | Hotels, bookings, steering rules |
| Booking tools | Lambda (7 functions) | Search, book, pay, confirm, cancel, validate |
| Graph query tool | Lambda (1 function, VPC) | Cypher queries to Neo4j knowledge graph |
| Tool routing | AgentCore Gateway (MCP) | Semantic tool discovery |
| Agent | AgentCore Runtime | Hosts the Strands agent with Bedrock |

This brings together the techniques from all previous modules:
- **Graph-RAG** (Module 1) → `query_knowledge_graph` Lambda connecting to Neo4j
- **Semantic tool routing** (Module 2) → AgentCore Gateway with MCP semantic search
- **Neurosymbolic guardrails** (Module 4) → Steering rules stored in DynamoDB
- **Steering** (Module 5) → Agent self-corrects based on STEER messages

## Workshop vs Self-paced

This notebook adapts to your environment:
- **At an AWS event:** Neo4j runs on your Code Editor EC2. The notebook detects the private IP and Security Group from CloudFormation outputs automatically.
- **Self-paced:** Set `NEO4J_HOST` manually to your Neo4j instance (AuraDB URI, local Docker, etc). If no Neo4j is available, the notebook deploys without the graph query tool.

## Region: **us-east-1**

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("✅ Environment ready")

---
## Step 0: Configuration

Set up AWS clients and naming conventions. All resources use a `workshop-` or `hotel-booking-` prefix so you can identify and clean them up later.

In [ ]:
import boto3
import json
import time
import zipfile
import io
import os
import subprocess

# --- Configuration ---
REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# Resource names (prefixed for easy cleanup)
HOTELS_TABLE = "workshop-Hotels"
BOOKINGS_TABLE = "workshop-Bookings"
STEERING_RULES_TABLE = "workshop-SteeringRules"
LAMBDA_ROLE_NAME = "workshop-LambdaExecutionRole"
AGENTCORE_ROLE_NAME = "workshop-AgentCoreExecutionRole"
GATEWAY_NAME = "HotelBookingGateway"
RUNTIME_NAME = "HotelBookingAgent"

# --- Workshop resource tag ---
# 08-cleanup/workshop_cleanup.py deletes a resource ONLY if it carries this exact
# key and value. A near-miss value is the same as no tag at all: cleanup reports
# UNTAGGED_BLOCKED, exits non-zero, and leaves billable resources running.
# The four shapes below are not interchangeable. Each service demands its own.
WORKSHOP_TAG_KEY = "WorkshopResource"
WORKSHOP_TAG_VALUE = "stop-ai-agent-hallucinations"
WORKSHOP_TAGS_MAP = {WORKSHOP_TAG_KEY: WORKSHOP_TAG_VALUE}                        # lambda, agentcore
WORKSHOP_TAGS_KV = [{"Key": WORKSHOP_TAG_KEY, "Value": WORKSHOP_TAG_VALUE}]       # dynamodb, iam, ecr
WORKSHOP_TAGS_KV_LOWER = [{"key": WORKSHOP_TAG_KEY, "value": WORKSHOP_TAG_VALUE}] # codebuild

# AWS clients
dynamodb = boto3.client("dynamodb", region_name=REGION)
dynamodb_resource = boto3.resource("dynamodb", region_name=REGION)
iam = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
agentcore_data = boto3.client("bedrock-agentcore", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
ec2 = boto3.client("ec2", region_name=REGION)

# --- Neo4j configuration from environment variables ---
# Workshop Studio: Pre-configured in /etc/environment by CloudFormation
# Self-paced: Set these before running the notebook
NEO4J_HOST = os.environ.get("NEO4J_HOST") or os.environ.get("NEO4J_URI", "").replace("bolt://", "").split(":")[0] or None
NEO4J_SECRET_ARN = os.environ.get("NEO4J_SECRET_ARN", "")
SECURITY_GROUP_ID = os.environ.get("SECURITY_GROUP_ID", "")

# Parse SUBNET_IDS from environment if available (comma-separated string)
subnet_ids_env = os.environ.get("SUBNET_IDS", "")
if subnet_ids_env:
    SUBNET_IDS = [s.strip() for s in subnet_ids_env.split(",") if s.strip()]
else:
    SUBNET_IDS = []

# Only get default VPC subnets if not already set from environment
if not SUBNET_IDS:
    try:
        vpcs = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"]
        if vpcs:
            subs = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpcs[0]["VpcId"]]}])["Subnets"]
            SUBNET_IDS = [s["SubnetId"] for s in subs[:2]]
    except Exception:
        pass

# --- Self-paced override ---
# If Neo4j is not detected automatically, set these manually:
# NEO4J_HOST = "your-neo4j-host"         # e.g., "localhost" or "xxxxx.databases.neo4j.io"
# NEO4J_SECRET_ARN = ""                   # Create a secret with your Neo4j password
# SECURITY_GROUP_ID = "sg-xxxxx"          # Your VPC security group
# SUBNET_IDS = ["subnet-xxxxx"]           # Your VPC subnets

print(f"Account:          {ACCOUNT_ID}")
print(f"Region:           {REGION}")
print(f"Neo4j Host:       {NEO4J_HOST or 'Not detected (Neo4j Lambda will be skipped)'}")
print(f"Neo4j Secret ARN: {NEO4J_SECRET_ARN or 'N/A'}")
print(f"Security Group:   {SECURITY_GROUP_ID or 'N/A'}")
print(f"Subnets:          {SUBNET_IDS or 'N/A'}")

### Recovery: Re-run safe

Every cell in this notebook is **idempotent** — if a resource already exists, the cell will find it and continue. You can re-run any cell or restart the notebook from any point without breaking anything.

If you need to recover variables from a previous partial run, execute the cell below to load existing resource IDs:

In [ ]:
# --- Recovery: Load existing resources if they already exist ---
# Run this cell if you restarted the kernel and need to recover state

# IAM Roles
try:
    LAMBDA_ROLE_ARN = iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
    print(f"Lambda role:    {LAMBDA_ROLE_ARN}")
except: 
    LAMBDA_ROLE_ARN = None
    print("Lambda role: not created yet")

try:
    AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
    print(f"AgentCore role: {AGENTCORE_ROLE_ARN}")
except:
    AGENTCORE_ROLE_ARN = None
    print("AgentCore role: not created yet")

# Lambda functions
if 'LAMBDA_ARNS' not in dir() or not LAMBDA_ARNS:
    LAMBDA_ARNS = {}
for tool_name in ["search_available_hotels", "book_hotel", "get_booking", "process_payment", "confirm_booking", "cancel_booking", "validate_booking_rules"]:
    try:
        resp = lambda_client.get_function(FunctionName=f"hotel-booking-{tool_name}")
        LAMBDA_ARNS[tool_name] = resp["Configuration"]["FunctionArn"]
    except: pass
if LAMBDA_ARNS:
    print(f"Lambda funcs:   {len(LAMBDA_ARNS)} found")
else:
    print("Lambda funcs:   not created yet")

# Gateway
GATEWAY_ID = None
try:
    gateways = agentcore.list_gateways()["items"]
    gw = next((g for g in gateways if g["name"] == GATEWAY_NAME), None)
    if gw:
        GATEWAY_ID = gw["gatewayId"]
        print(f"Gateway:        {GATEWAY_ID} ({gw['status']})")
except: 
    print("Gateway:        not created yet")

# Runtime
RUNTIME_ID = None
RUNTIME_ARN = None
try:
    runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]
    rt = next((r for r in runtimes if r["agentRuntimeName"] == RUNTIME_NAME), None)
    if rt:
        RUNTIME_ID = rt["agentRuntimeId"]
        RUNTIME_ARN = rt["agentRuntimeArn"]
        print(f"Runtime:        {RUNTIME_ID} ({rt['status']})")
except:
    print("Runtime:        not created yet")

print("\nRecovery complete. You can now run any step.")

---
## Step 1: Create DynamoDB Tables

The booking system uses three tables:

| Table | Key | What it stores |
|-------|-----|----------------|
| **workshop-Hotels** | `hotel_id` | Hotel catalog: name, city, stars, price, availability |
| **workshop-Bookings** | `booking_id` | Reservations: guest, dates, status (PENDING → PAID → CONFIRMED) |
| **workshop-SteeringRules** | `rule_id` | Business rules the agent must follow (max guests, advance booking, etc.) |

All tables use **PAY_PER_REQUEST** billing — you only pay for what you use during the workshop.

In [ ]:
tables = [
    {"name": HOTELS_TABLE, "key": "hotel_id"},
    {"name": BOOKINGS_TABLE, "key": "booking_id"},
    {"name": STEERING_RULES_TABLE, "key": "rule_id"},
]

for t in tables:
    try:
        dynamodb.create_table(
            TableName=t["name"],
            KeySchema=[{"AttributeName": t["key"], "KeyType": "HASH"}],
            AttributeDefinitions=[{"AttributeName": t["key"], "AttributeType": "S"}],
            BillingMode="PAY_PER_REQUEST",
            Tags=WORKSHOP_TAGS_KV,
        )
        print(f"Creating {t['name']}...")
    except dynamodb.exceptions.ResourceInUseException:
        # Pre-existing table from an earlier run. Tag it anyway, otherwise
        # cleanup blocks on it and this notebook leaves billable state behind.
        table_arn = dynamodb.describe_table(TableName=t["name"])["Table"]["TableArn"]
        dynamodb.tag_resource(ResourceArn=table_arn, Tags=WORKSHOP_TAGS_KV)
        print(f"{t['name']} already exists — tagged")

# Wait for all tables to be active
for t in tables:
    waiter = dynamodb.get_waiter("table_exists")
    waiter.wait(TableName=t["name"])
    print(f"{t['name']} — ACTIVE")

print("\nAll tables ready.")

---
## Step 2: Seed Hotel Data

We populate the Hotels table with **18 hotels** across 18 cities worldwide, with prices derived from the hotel-bookings.csv dataset average daily rates by country.

Notice that **AnyCompany Rome Centro** has `available_rooms: 0` — this tests the "sold out" scenario where the agent must handle unavailability gracefully.

In [ ]:
HOTELS = [
    {"hotel_id": "anycompany-lisbon-resort", "name": "AnyCompany Lisbon Resort", "city": "Lisbon", "country": "Portugal", "stars": 4, "price_per_night": 95, "max_guests_per_room": 4, "total_rooms": 80, "available_rooms": 20},
    {"hotel_id": "anycompany-london-city", "name": "AnyCompany London City Hotel", "city": "London", "country": "UK", "stars": 4, "price_per_night": 97, "max_guests_per_room": 3, "total_rooms": 120, "available_rooms": 15},
    {"hotel_id": "anycompany-paris-central", "name": "AnyCompany Paris Central", "city": "Paris", "country": "France", "stars": 5, "price_per_night": 110, "max_guests_per_room": 4, "total_rooms": 100, "available_rooms": 12},
    {"hotel_id": "anycompany-barcelona-beach", "name": "AnyCompany Barcelona Beach", "city": "Barcelona", "country": "Spain", "stars": 4, "price_per_night": 118, "max_guests_per_room": 3, "total_rooms": 60, "available_rooms": 25},
    {"hotel_id": "anycompany-berlin-mitte", "name": "AnyCompany Berlin Mitte", "city": "Berlin", "country": "Germany", "stars": 3, "price_per_night": 105, "max_guests_per_room": 3, "total_rooms": 70, "available_rooms": 18},
    {"hotel_id": "anycompany-rome-centro", "name": "AnyCompany Rome Centro", "city": "Rome", "country": "Italy", "stars": 3, "price_per_night": 115, "max_guests_per_room": 2, "total_rooms": 45, "available_rooms": 0},
    {"hotel_id": "anycompany-dublin-temple", "name": "AnyCompany Dublin Temple Bar", "city": "Dublin", "country": "Ireland", "stars": 4, "price_per_night": 100, "max_guests_per_room": 3, "total_rooms": 50, "available_rooms": 10},
    {"hotel_id": "anycompany-brussels-grand", "name": "AnyCompany Brussels Grand Place", "city": "Brussels", "country": "Belgium", "stars": 3, "price_per_night": 114, "max_guests_per_room": 2, "total_rooms": 40, "available_rooms": 8},
    {"hotel_id": "anycompany-sao-paulo-jardins", "name": "AnyCompany São Paulo Jardins", "city": "São Paulo", "country": "Brazil", "stars": 4, "price_per_night": 112, "max_guests_per_room": 3, "total_rooms": 90, "available_rooms": 22},
    {"hotel_id": "anycompany-amsterdam-canal", "name": "AnyCompany Amsterdam Canal", "city": "Amsterdam", "country": "Netherlands", "stars": 4, "price_per_night": 108, "max_guests_per_room": 2, "total_rooms": 55, "available_rooms": 14},
    {"hotel_id": "anycompany-new-york-midtown", "name": "AnyCompany New York Midtown", "city": "New York", "country": "USA", "stars": 5, "price_per_night": 124, "max_guests_per_room": 4, "total_rooms": 150, "available_rooms": 5},
    {"hotel_id": "anycompany-zurich-lake", "name": "AnyCompany Zurich Lake", "city": "Zurich", "country": "Switzerland", "stars": 5, "price_per_night": 122, "max_guests_per_room": 3, "total_rooms": 60, "available_rooms": 9},
    {"hotel_id": "anycompany-beijing-imperial", "name": "AnyCompany Beijing Imperial", "city": "Beijing", "country": "China", "stars": 4, "price_per_night": 109, "max_guests_per_room": 3, "total_rooms": 110, "available_rooms": 30},
    {"hotel_id": "anycompany-vienna-ring", "name": "AnyCompany Vienna Ring", "city": "Vienna", "country": "Austria", "stars": 4, "price_per_night": 107, "max_guests_per_room": 3, "total_rooms": 65, "available_rooms": 16},
    {"hotel_id": "anycompany-stockholm-old", "name": "AnyCompany Stockholm Old Town", "city": "Stockholm", "country": "Sweden", "stars": 3, "price_per_night": 114, "max_guests_per_room": 2, "total_rooms": 35, "available_rooms": 7},
    {"hotel_id": "anycompany-shanghai-bund", "name": "AnyCompany Shanghai Bund", "city": "Shanghai", "country": "China", "stars": 5, "price_per_night": 111, "max_guests_per_room": 4, "total_rooms": 130, "available_rooms": 28},
    {"hotel_id": "anycompany-warsaw-royal", "name": "AnyCompany Warsaw Royal", "city": "Warsaw", "country": "Poland", "stars": 3, "price_per_night": 108, "max_guests_per_room": 2, "total_rooms": 50, "available_rooms": 12},
    {"hotel_id": "anycompany-porto-riverside", "name": "AnyCompany Porto Riverside", "city": "Porto", "country": "Portugal", "stars": 3, "price_per_night": 75, "max_guests_per_room": 2, "total_rooms": 30, "available_rooms": 15},
]

hotels_table = dynamodb_resource.Table(HOTELS_TABLE)

# Seed all hotels
for hotel in HOTELS:
    hotels_table.put_item(Item=hotel)

# Show sample of first 3 hotels
print(f"Seeding {len(HOTELS)} hotels...")
print("\nSample hotels:")
for hotel in HOTELS[:3]:
    print(f"  {hotel['name']:30s} | {hotel['city']:15s} | {'★' * hotel['stars']:5s} | ${hotel['price_per_night']}/night | {hotel['available_rooms']} available")
print(f"  ... ({len(HOTELS) - 3} more)")

print(f"\n✓ {len(HOTELS)} hotels seeded successfully")

## Step 3: Seed Steering Rules

Steering rules are the **production version of the neurosymbolic guardrails** from Module 3. Instead of being hardcoded in Python, they live in DynamoDB — you can change them without redeploying the agent.

Each rule has:
- **`fail_message`**: What the agent reports when the rule is violated
- **`steer_message`**: How the agent should self-correct (the STEER instruction from Module 4)

| Rule | Action | Condition | What Happens |
|------|--------|-----------|-------------|
| max-guests | book | guests > 10 | STEER: adjust to 10 guests |
| valid-dates | book | nights < 1 | STEER: swap check-in/check-out |
| advance-booking | book | same-day | STEER: move to tomorrow |
| payment-before-confirm | confirm | status != PAID | STEER: process payment first |
| cancellation-window | cancel | < 48h before check-in | STEER: offer modification |
| already-cancelled | cancel | status == CANCELLED | STEER: offer new reservation |

In [ ]:
STEERING_RULES = [
    {"rule_id": "max-guests", "action": "book", "condition_field": "guests", "operator": "gt", "threshold": 10,
     "fail_message": "Guest count exceeds maximum of 10",
     "steer_message": "The booking exceeds the hotel maximum of 10 guests per room. Calculate how many rooms are needed (divide {guests} by 10, rounding up). Explain to the user that you are splitting the reservation into multiple rooms, then immediately call book_hotel multiple times: fill rooms with 10 guests each until all guests are accommodated. After all calls succeed, confirm the split reservation with room details (e.g., Reserved 2 rooms: 10 + 5 guests).",
     "enabled": True},
    {"rule_id": "valid-dates", "action": "book", "condition_field": "nights", "operator": "lt", "threshold": 1,
     "fail_message": "Check-in date must be before check-out date",
     "steer_message": "The dates appear reversed. Swap check-in and check-out, proceed with the booking, and tell the user.",
     "enabled": True},
    {"rule_id": "advance-booking", "action": "book", "condition_field": "days_until_checkin", "operator": "lt", "threshold": 1,
     "fail_message": "Booking must be made at least 1 day in advance",
     "steer_message": "Same-day bookings are not available. Adjust the check-in to tomorrow's date and proceed.",
     "enabled": True},
    {"rule_id": "payment-before-confirm", "action": "confirm", "condition_field": "booking_status", "operator": "ne", "threshold": "PAID",
     "fail_message": "Payment must be processed before confirmation",
     "steer_message": "Confirmation requires payment first. Process the payment, then confirm.",
     "enabled": True},
    {"rule_id": "cancellation-window", "action": "cancel", "condition_field": "days_until_checkin", "operator": "lt", "threshold": 2,
     "fail_message": "Cannot cancel within 48 hours of check-in",
     "steer_message": "Cancellation within 48 hours is not available. Offer to modify the reservation instead.",
     "enabled": True},
    {"rule_id": "already-cancelled", "action": "cancel", "condition_field": "booking_status", "operator": "eq", "threshold": "CANCELLED",
     "fail_message": "Booking is already cancelled",
     "steer_message": "This booking is already cancelled. Offer to make a new reservation.",
     "enabled": True},
]

rules_table = dynamodb_resource.Table(STEERING_RULES_TABLE)

# Seed all steering rules
for rule in STEERING_RULES:
    rules_table.put_item(Item=rule)

# Show summary of rules
print(f"Seeding {len(STEERING_RULES)} steering rules...")
print("\nRules by action:")
for action in ["book", "confirm", "cancel"]:
    rules_for_action = [r for r in STEERING_RULES if r["action"] == action]
    print(f"  {action:8s} → {len(rules_for_action)} rules")

print(f"\n✓ {len(STEERING_RULES)} steering rules seeded successfully")

---
## Step 4: Create IAM Roles

We need two IAM roles:

1. **Lambda Execution Role** — allows the 7 Lambda functions to read/write DynamoDB and write CloudWatch logs
2. **AgentCore Execution Role** — allows the agent runtime to invoke Bedrock models, call Lambda tools via the Gateway, and read bookings for guardrail validation

Both roles use **least-privilege** policies scoped to the specific resources created in this workshop.

In [ ]:
# --- Lambda Execution Role ---
# This role is used by ALL Lambda functions (booking tools + Neo4j query).
# It includes DynamoDB access, CloudWatch logs, VPC networking (for Neo4j Lambda),
# and Secrets Manager access (to retrieve the Neo4j password).

lambda_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

lambda_statements = [
    {
        "Effect": "Allow",
        "Action": ["dynamodb:GetItem", "dynamodb:PutItem", "dynamodb:UpdateItem", "dynamodb:Scan", "dynamodb:Query"],
        "Resource": [
            f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{HOTELS_TABLE}",
            f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{BOOKINGS_TABLE}",
            f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{STEERING_RULES_TABLE}",
        ]
    },
    {
        "Effect": "Allow",
        "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
        "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/lambda/hotel-booking-*"
    },
]

# Add VPC networking permissions (needed for Neo4j Lambda to connect to EC2)
if NEO4J_HOST and SECURITY_GROUP_ID:
    lambda_statements.append({
        "Effect": "Allow",
        "Action": [
            "ec2:CreateNetworkInterface",
            "ec2:DescribeNetworkInterfaces",
            "ec2:DeleteNetworkInterface",
        ],
        "Resource": "*"
    })

# Add Secrets Manager access (Neo4j password)
if NEO4J_SECRET_ARN:
    lambda_statements.append({
        "Effect": "Allow",
        "Action": ["secretsmanager:GetSecretValue"],
        "Resource": NEO4J_SECRET_ARN
    })

lambda_policy = json.dumps({"Version": "2012-10-17", "Statement": lambda_statements})

try:
    iam.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=lambda_trust,
        Description="Lambda role for hotel booking tools",
        Tags=WORKSHOP_TAGS_KV,
    )
    print(f"Created {LAMBDA_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    # Named role only. Nothing here enumerates roles or matches on name shape,
    # so the shared AmazonBedrockAgentCoreSDKCodeBuild-* role is never touched.
    iam.tag_role(RoleName=LAMBDA_ROLE_NAME, Tags=WORKSHOP_TAGS_KV)
    print(f"{LAMBDA_ROLE_NAME} already exists — tagged, updating policy")

iam.put_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyName="DynamoDBAndLogs", PolicyDocument=lambda_policy)
iam.attach_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")

LAMBDA_ROLE_ARN = iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
print(f"ARN: {LAMBDA_ROLE_ARN}")
if NEO4J_HOST:
    print(f"  + VPC networking permissions (Neo4j Lambda)")
    print(f"  + Secrets Manager access ({NEO4J_SECRET_ARN})")

In [ ]:
# --- AgentCore Execution Role ---

agentcore_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

# Module 7 deploys an AgentCore Memory resource named workshop_HotelBookingMemory and
# runs a second agent against it. AgentCore mints memory ids as "<name>-<suffix>", so the
# memory ARN is scoped by that name prefix rather than granted as Resource: "*".
#
# Without these actions the memory-enabled runtime gets AccessDeniedException on ListEvents
# and every invoke_agent_runtime call returns HTTP 500. Module 7 could never have worked
# with the gateway-only permission set that used to be here. See bug B41.
MEMORY_NAME = "workshop_HotelBookingMemory"
MEMORY_ARN = f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:memory/{MEMORY_NAME}-*"

agentcore_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["lambda:InvokeFunction"], "Resource": f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:hotel-booking-*"},
        {"Effect": "Allow", "Action": ["dynamodb:GetItem"], "Resource": f"arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{BOOKINGS_TABLE}"},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/*"},
        {"Effect": "Allow", "Action": ["bedrock-agentcore:GetGateway", "bedrock-agentcore:GetGatewayTarget", "bedrock-agentcore:ListGatewayTargets", "bedrock-agentcore:InvokeGateway"], "Resource": "*"},
        # AgentCoreMemorySessionManager calls CreateEvent / ListEvents / GetEvent /
        # DeleteEvent / RetrieveMemoryRecords. GetMemory is the control-plane validation
        # the SDK performs on startup. Scoped to the workshop memory, not "*".
        {"Effect": "Allow", "Action": [
            "bedrock-agentcore:CreateEvent",
            "bedrock-agentcore:GetEvent",
            "bedrock-agentcore:ListEvents",
            "bedrock-agentcore:DeleteEvent",
            "bedrock-agentcore:ListSessions",
            "bedrock-agentcore:ListActors",
            "bedrock-agentcore:RetrieveMemoryRecords",
            "bedrock-agentcore:ListMemoryRecords",
            "bedrock-agentcore:GetMemoryRecord",
            "bedrock-agentcore:GetMemory",
        ], "Resource": [MEMORY_ARN, f"{MEMORY_ARN}/*"]},
        {"Effect": "Allow", "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"], "Resource": f"arn:aws:ecr:{REGION}:{ACCOUNT_ID}:repository/bedrock-agentcore-*"},
    ]
})

try:
    iam.create_role(
        RoleName=AGENTCORE_ROLE_NAME,
        AssumeRolePolicyDocument=agentcore_trust,
        Description="AgentCore role for hotel booking agent",
        Tags=WORKSHOP_TAGS_KV,
    )
    print(f"Created {AGENTCORE_ROLE_NAME}")
except iam.exceptions.EntityAlreadyExistsException:
    # Named role only, same reasoning as the Lambda role above.
    iam.tag_role(RoleName=AGENTCORE_ROLE_NAME, Tags=WORKSHOP_TAGS_KV)
    print(f"{AGENTCORE_ROLE_NAME} already exists — tagged, updating policy")

# Always update the policy to ensure ECR permissions are included
iam.put_role_policy(RoleName=AGENTCORE_ROLE_NAME, PolicyName="AgentCorePermissions", PolicyDocument=agentcore_policy)
iam.attach_role_policy(RoleName=AGENTCORE_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/AWSXRayDaemonWriteAccess")

AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
print(f"ARN: {AGENTCORE_ROLE_ARN}")

print("\nWaiting 10s for IAM propagation...")
time.sleep(10)
print("Ready.")

---
## Step 5: Deploy Lambda Functions

Each Lambda function is a **booking tool** that the agent can call through the Gateway:

| Function | What it does | Key DynamoDB operations |
|----------|-------------|------------------------|
| `search_available_hotels` | Find hotels by city, country, price, stars | Scan with filters |
| `book_hotel` | Create a reservation (status: PENDING) | PutItem + UpdateItem (decrement rooms) |
| `get_booking` | Retrieve booking details | GetItem |
| `process_payment` | Pay for a booking (PENDING → PAID) | UpdateItem |
| `confirm_booking` | Confirm after payment (PAID → CONFIRMED) | UpdateItem |
| `cancel_booking` | Cancel and return room to inventory | UpdateItem (both tables) |
| `validate_booking_rules` | Check steering rules from DynamoDB | Scan rules + evaluate |

The code for each function lives in `lambda_tools/<name>/lambda_function.py`. We zip each one and deploy it.

In [ ]:
LAMBDA_TOOLS = [
    "search_available_hotels", "book_hotel", "get_booking",
    "process_payment", "confirm_booking", "cancel_booking",
    "validate_booking_rules",
]

TOOL_DESCRIPTIONS = {
    "search_available_hotels": "Search hotels by city, country, price range, or star rating",
    "book_hotel": "Create a hotel reservation with pending payment status",
    "get_booking": "Retrieve booking details by booking ID",
    "process_payment": "Process payment for a pending booking",
    "confirm_booking": "Confirm a paid booking",
    "cancel_booking": "Cancel an existing booking and return room to inventory",
    "validate_booking_rules": "Validate booking against business rules and steering policies",
}

if 'LAMBDA_ARNS' not in dir() or not LAMBDA_ARNS:
    LAMBDA_ARNS = {}

for tool_name in LAMBDA_TOOLS:
    function_name = f"hotel-booking-{tool_name}"

    # Zip the Lambda code
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(f"lambda_tools/{tool_name}/lambda_function.py", "lambda_function.py")
    zip_bytes = zip_buffer.getvalue()

    try:
        resp = lambda_client.create_function(
            FunctionName=function_name,
            Description=TOOL_DESCRIPTIONS.get(tool_name, ""),
            Runtime="python3.11",
            Role=LAMBDA_ROLE_ARN,
            Handler="lambda_function.handler",
            Code={"ZipFile": zip_bytes},
            Timeout=30,
            MemorySize=256,
            Environment={"Variables": {
                "HOTELS_TABLE": HOTELS_TABLE,
                "BOOKINGS_TABLE": BOOKINGS_TABLE,
                "STEERING_RULES_TABLE": STEERING_RULES_TABLE,
            }},
            Tags=WORKSHOP_TAGS_MAP,  # Lambda takes a dict here, not a [{Key, Value}] list
        )
        LAMBDA_ARNS[tool_name] = resp["FunctionArn"]
        print(f"  Created  {function_name}")
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_bytes)
        resp = lambda_client.get_function(FunctionName=function_name)
        LAMBDA_ARNS[tool_name] = resp["Configuration"]["FunctionArn"]
        lambda_client.tag_resource(Resource=LAMBDA_ARNS[tool_name], Tags=WORKSHOP_TAGS_MAP)
        print(f"  Updated  {function_name} — tagged")

# Wait for all to be active
for tool_name in LAMBDA_TOOLS:
    waiter = lambda_client.get_waiter("function_active_v2")
    waiter.wait(FunctionName=f"hotel-booking-{tool_name}")

print(f"\n{len(LAMBDA_ARNS)} Lambda functions deployed and active.")

### Deploy Neo4j Query Lambda (VPC)

The `query_knowledge_graph` Lambda connects to the Neo4j instance running on the Code Editor EC2.
Unlike the booking Lambdas, this one runs **inside the VPC** so it can reach the EC2 private IP on port 7687.

It also needs the `neo4j` Python driver as a dependency, which we package as a Lambda Layer.

:::alert{type="info" header="Why VPC Lambda?"}
**Workshop architecture:** Neo4j runs on EC2 with a **private IP** (10.0.x.x) inside a VPC for security. Lambda functions by default run outside VPCs and cannot reach private IPs.

**VPC Lambda requirements:**
- Elastic Network Interfaces (ENI) attached to Lambda
- Security group rules allowing Lambda → Neo4j on port 7687
- Subnets in the same VPC as Neo4j
- **Deployment time:** 2-5 minutes (ENI creation + attachment)

**Simpler alternatives for production:**
1. **Neo4j AuraDB** (managed cloud) — public endpoint with TLS, no VPC needed
2. **Public NLB** — expose Neo4j with internet-facing load balancer (security risk)
3. **Embedded Neo4j** — run Neo4j in the same AgentCore Runtime container

This workshop uses VPC Lambda because it demonstrates real-world enterprise patterns where databases are private and isolated.
:::

In [ ]:
import sys
# Deploy Neo4j query Lambda — only if Neo4j infrastructure was detected
if NEO4J_HOST and SECURITY_GROUP_ID and SUBNET_IDS:
    print("Deploying Neo4j query Lambda with VPC config...")

    # Step 1: Create a Lambda Layer with the neo4j Python driver
    # The neo4j package is not in the Lambda runtime — we need to bundle it
    import shutil, subprocess

    layer_dir = "/tmp/neo4j-layer/python"
    if os.path.exists("/tmp/neo4j-layer"):
        shutil.rmtree("/tmp/neo4j-layer", ignore_errors=True)
    os.makedirs(layer_dir)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "neo4j", "-t", layer_dir, "--quiet"],
        check=True, capture_output=True,
    )

    layer_zip = io.BytesIO()
    with zipfile.ZipFile(layer_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk("/tmp/neo4j-layer"):
            for f in files:
                fp = os.path.join(root, f)
                arcname = os.path.relpath(fp, "/tmp/neo4j-layer")
                zf.write(fp, arcname)
    layer_bytes = layer_zip.getvalue()
    print(f"  Neo4j layer: {len(layer_bytes) / 1024 / 1024:.1f} MB")

    # Create or update the Lambda Layer
    try:
        layer_resp = lambda_client.publish_layer_version(
            LayerName="workshop-neo4j-driver",
            Description="Neo4j Python driver for query_knowledge_graph Lambda",
            Content={"ZipFile": layer_bytes},
            CompatibleRuntimes=["python3.11", "python3.12", "python3.13"],
            # PublishLayerVersion has no Tags member. AWS does not support tags on
            # layer versions at all, which is why workshop_cleanup.py lists
            # "lambda-layer-version" in UNTAGGABLE_KINDS and deletes it by exact name.
        )
        LAYER_ARN = layer_resp["LayerVersionArn"]
        print(f"  Layer: {LAYER_ARN}")
    except Exception as e:
        print(f"  Layer error: {e}")
        LAYER_ARN = None

    # Step 2: Deploy the Lambda function with VPC config
    function_name = "hotel-booking-query_knowledge_graph"
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("lambda_tools/query_knowledge_graph/lambda_function.py", "lambda_function.py")
    zip_bytes = zip_buffer.getvalue()

    lambda_config = {
        "FunctionName": function_name,
        "Description": "Execute Cypher queries against Neo4j knowledge graph (VPC)",
        "Runtime": "python3.11",
        "Role": LAMBDA_ROLE_ARN,
        "Handler": "lambda_function.handler",
        "Code": {"ZipFile": zip_bytes},
        "Timeout": 30,
        "MemorySize": 256,
        "Environment": {"Variables": {
            "NEO4J_HOST": NEO4J_HOST,
            "NEO4J_PASSWORD_SECRET_ARN": NEO4J_SECRET_ARN,
        }},
        "VpcConfig": {
            "SubnetIds": SUBNET_IDS,
            "SecurityGroupIds": [SECURITY_GROUP_ID],
        },
        "Tags": WORKSHOP_TAGS_MAP,  # dict, matching create_function above
    }
    if LAYER_ARN:
        lambda_config["Layers"] = [LAYER_ARN]

    try:
        resp = lambda_client.create_function(**lambda_config)
        LAMBDA_ARNS["query_knowledge_graph"] = resp["FunctionArn"]
        print(f"  Created {function_name} (VPC: {SECURITY_GROUP_ID})")
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_bytes)
        resp = lambda_client.get_function(FunctionName=function_name)
        LAMBDA_ARNS["query_knowledge_graph"] = resp["Configuration"]["FunctionArn"]
        lambda_client.tag_resource(Resource=LAMBDA_ARNS["query_knowledge_graph"], Tags=WORKSHOP_TAGS_MAP)
        print(f"  Updated {function_name} — tagged")

    # Wait for it to be active
    waiter = lambda_client.get_waiter("function_active_v2")
    waiter.wait(FunctionName=function_name)
    print(f"  {function_name} — ACTIVE")

else:
    print("Neo4j not detected — skipping query_knowledge_graph Lambda.")
    print("The agent will work with booking tools only (no graph queries).")
    print("To enable: set NEO4J_HOST, NEO4J_SECRET_ARN, SECURITY_GROUP_ID, SUBNET_IDS")

print(f"\nTotal Lambda functions: {len(LAMBDA_ARNS)}")

---
## Step 6: Create AgentCore Gateway

The **AgentCore Gateway** is the production version of the FAISS-based semantic tool filtering from Module 1.

Instead of building a local FAISS index, the Gateway uses **MCP (Model Context Protocol)** with built-in **semantic search** to route requests to the right Lambda tool based on the user's query.

For example:
- "Find me a hotel in Paris" → routes to `search_available_hotels`
- "Pay for my booking" → routes to `process_payment`
- "Is my booking valid?" → routes to `validate_booking_rules`

In [ ]:
try:
    resp = agentcore.create_gateway(
        name=GATEWAY_NAME,
        description="Hotel booking tools: search, book, pay, confirm, cancel, and validate business rules.",
        protocolType="MCP",
        protocolConfiguration={
            "mcp": {
                "instructions": "Hotel booking tools: search, book, pay, confirm, cancel, and validate business rules.",
                "searchType": "SEMANTIC",
                "supportedVersions": ["2025-03-26"],
            }
        },
        authorizerType="NONE",
        roleArn=AGENTCORE_ROLE_ARN,
        tags=WORKSHOP_TAGS_MAP,  # AgentCore: lowercase tags=, map shape
    )
    GATEWAY_ID = resp["gatewayId"]
    print(f"Created gateway: {GATEWAY_ID}")
except Exception as e:
    if "ConflictException" in str(type(e)) or "already exists" in str(e).lower():
        gateways = agentcore.list_gateways()["items"]
        GATEWAY_ID = next(g["gatewayId"] for g in gateways if g["name"] == GATEWAY_NAME)
        # GatewaySummary carries no arn, so fetch it before tagging.
        gateway_arn = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)["gatewayArn"]
        agentcore.tag_resource(resourceArn=gateway_arn, tags=WORKSHOP_TAGS_MAP)
        print(f"Gateway already exists: {GATEWAY_ID} — tagged")
    else:
        raise

# Wait for gateway to be ready
print("Waiting for gateway...")
for _ in range(30):
    gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = gw.get("status", "UNKNOWN")
    if status in ("READY", "ACTIVE", "CREATE_COMPLETE"):
        break
    time.sleep(5)

print(f"Gateway status: {status}")
print(f"Gateway ID: {GATEWAY_ID}")

## Step 7: Register Lambda Tools as Gateway Targets

Each Lambda function is registered as a **Gateway Target** with its tool schema. The Gateway uses these schemas for **semantic routing** — matching user queries to the right tool automatically.

All tool schemas are loaded from `tool_schemas/tools.json`. If Neo4j was detected, the `query_knowledge_graph` tool is included; otherwise it is skipped.

In [ ]:
# Gateway targets carry no tags: CreateGatewayTarget has no tags member in the
# API. They are deleted as children of the gateway, so workshop_cleanup.py
# never tag-checks them.

# Load all tool schemas
with open("tool_schemas/tools.json") as f:
    all_schemas = json.load(f)

# Include query_knowledge_graph only if we deployed its Lambda
available_tools = set(LAMBDA_ARNS.keys())
tool_schemas = {s["name"]: s for s in all_schemas if s["name"] in available_tools}

print(f"Registering {len(tool_schemas)} tools as Gateway targets:")
for name in tool_schemas:
    print(f"  - {name}" + (" (Neo4j VPC)" if name == "query_knowledge_graph" else ""))

# Clean schema properties to only include fields the API accepts
ALLOWED_PROP_FIELDS = {"type", "description", "properties", "required", "items"}

def clean_schema(schema_obj):
    """Recursively clean schema to only include API-accepted fields."""
    if not isinstance(schema_obj, dict):
        return schema_obj
    cleaned = {}
    for k, v in schema_obj.items():
        if k == "properties" and isinstance(v, dict):
            cleaned[k] = {pk: clean_schema(pv) for pk, pv in v.items()}
        elif k in ALLOWED_PROP_FIELDS:
            cleaned[k] = clean_schema(v) if isinstance(v, dict) else v
    return cleaned

print()
for tool_name, lambda_arn in LAMBDA_ARNS.items():
    target_name = tool_name.replace("_", "-")
    schema = tool_schemas.get(tool_name)
    if not schema:
        continue

    cleaned_input = clean_schema(schema["input_schema"])

    try:
        agentcore.create_gateway_target(
            name=target_name,
            gatewayIdentifier=GATEWAY_ID,  # Fixed: was gatewayId, should be gatewayIdentifier
            credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
            targetConfiguration={
                "mcp": {
                    "lambda": {
                        "lambdaArn": lambda_arn,
                        "toolSchema": {
                            "inlinePayload": [{
                                "name": schema["name"],
                                "description": schema["description"],
                                "inputSchema": cleaned_input,
                            }]
                        },
                    }
                }
            },
        )
        print(f"  Registered: {target_name}")
    except Exception as e:
        if "ConflictException" in str(type(e)) or "already exists" in str(e).lower():
            print(f"  Already exists: {target_name}")
        else:
            print(f"  ERROR: {target_name}: {e}")
            raise

print(f"\n{len(LAMBDA_ARNS)} gateway targets configured.")

---
## Step 8: Deploy AgentCore Runtime

Now we deploy the **Strands agent** to AgentCore using the `bedrock-agentcore-starter-toolkit`.

The toolkit handles:
- Building the deployment package for ARM64 architecture
- Creating the ECR repository and Docker image
- Creating the AgentCore Runtime with proper configuration

The agent code (`booking_agent.py`) connects to the Gateway via MCP, uses Bedrock Claude as the model, and includes hard guardrails via lifecycle hooks.

> **This step takes 3-5 minutes.** The toolkit builds a container image and deploys it to AgentCore.

In [ ]:
# Pre-flight cleanup: remove the starter toolkit config from previous runs.
# The CodeBuild project is intentionally NOT deleted here. The starter toolkit's
# launch() calls create_or_update_project, which updates an existing project
# instead of raising ResourceAlreadyExistsException, so the deploy re-runs
# idempotently without a blind delete. A blind delete_project also bypassed the
# WorkshopResource tag gate that every other teardown path here honors. See
# verify.md V7.
import os

# Delete starter toolkit config with old runtime ID.
# Remove only the config file THIS notebook's toolkit run writes, in THIS
# directory. The previous version globbed ~/.bedrock_agentcore*.yaml. HOME is
# shared with every other AgentCore project on the machine, so running this
# workshop destroyed unrelated local config. Exact path, current directory only.
# See bug B45.
local_cfg = os.path.join(os.getcwd(), ".bedrock_agentcore.yaml")
if os.path.exists(local_cfg):
    os.remove(local_cfg)
    print(f"Deleted stale config: {local_cfg}")

print("✅ Pre-flight cleanup done")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Recover role ARN
AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
print(f"Role: {AGENTCORE_ROLE_ARN}")

# Get Gateway URL
if not GATEWAY_ID:
    raise ValueError("GATEWAY_ID not set. Run the recovery cell or the gateway creation cell first.")
gw_info = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
GATEWAY_URL = gw_info["gatewayUrl"]
print(f"Gateway URL: {GATEWAY_URL}")

# Configure the agent
agent_runtime = Runtime()

agent_runtime.configure(
    entrypoint="booking_agent.py",
    execution_role=AGENTCORE_ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="agent_requirements.txt",
    region=REGION,
    agent_name=RUNTIME_NAME,
    deployment_type="container",
    non_interactive=True,
)

print("\nLaunching agent (3-5 minutes)...")

result = agent_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "AWS_REGION": REGION,
        "BOOKINGS_TABLE": BOOKINGS_TABLE,
        "GATEWAY_URL": GATEWAY_URL,
    },
)

RUNTIME_ARN = result.agent_arn
RUNTIME_ID = RUNTIME_ARN.split("/")[-1] if RUNTIME_ARN else None
print(f"\nAgent deployed: {RUNTIME_ARN}")

---

### Tag the toolkit-created resources

The starter toolkit creates the ECR repository, the CodeBuild project and the AgentCore
Runtime on your behalf, and does not pass the workshop tag through. Module 8 deletes only
tagged resources, so these three have to be tagged now. Skipping this cell means teardown
will refuse to delete them and you will keep paying for them.

In [ ]:
# --- Tag the resources the starter toolkit created ---
# The toolkit creates the ECR repo, the CodeBuild project and the AgentCore Runtime
# itself and does not forward tags, so they are tagged here, immediately after deploy.
# Cleanup deletes only tagged resources; skip this cell and teardown will refuse to
# remove them and will exit non-zero, leaving billable infrastructure running.
#
# Every target below is addressed by EXACT name or by ARN. Nothing is enumerated and
# nothing is prefix-matched. In particular the toolkit's shared
# AmazonBedrockAgentCoreSDKCodeBuild-* IAM role is deliberately NOT tagged: it is
# shared across projects, costs nothing, and tagging it would make it eligible for
# deletion. Deleting roles by name shape is bug B6, which already destroyed five
# unrelated roles in this account.

ecr_client = boto3.client("ecr", region_name=REGION)
codebuild_client = boto3.client("codebuild", region_name=REGION)

ECR_REPO = f"bedrock-agentcore-{RUNTIME_NAME.lower()}"
CB_PROJECT = f"bedrock-agentcore-{RUNTIME_NAME.lower()}-builder"

# ECR repository — resourceArn=, but capitalised {Key, Value} members
try:
    repo = ecr_client.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
    ecr_client.tag_resource(resourceArn=repo["repositoryArn"], tags=WORKSHOP_TAGS_KV)
    print(f"Tagged ECR repository: {ECR_REPO}")
except ecr_client.exceptions.RepositoryNotFoundException:
    print(f"ECR repository not found (nothing to tag): {ECR_REPO}")

# CodeBuild project — lowercase key/value members, applied via update_project.
# update_project REPLACES the whole tag set, so merge rather than clobber whatever
# the starter toolkit put there.
projects = codebuild_client.batch_get_projects(names=[CB_PROJECT])["projects"]
if projects:
    merged = [t for t in projects[0].get("tags", []) if t.get("key") != WORKSHOP_TAG_KEY]
    codebuild_client.update_project(name=CB_PROJECT, tags=merged + WORKSHOP_TAGS_KV_LOWER)
    print(f"Tagged CodeBuild project: {CB_PROJECT}")
else:
    print(f"CodeBuild project not found (nothing to tag): {CB_PROJECT}")

# AgentCore Runtime — tag by the ARN the launch returned
if not RUNTIME_ARN:
    raise RuntimeError("RUNTIME_ARN is not set. Re-run the launch cell before tagging.")
agentcore.tag_resource(resourceArn=RUNTIME_ARN, tags=WORKSHOP_TAGS_MAP)
print(f"Tagged AgentCore Runtime: {RUNTIME_ARN}")

# Verify rather than trust: read the tags back.
runtime_tags = agentcore.list_tags_for_resource(resourceArn=RUNTIME_ARN).get("tags", {})
if runtime_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
    raise RuntimeError(f"Runtime tag did not stick. Read back: {runtime_tags}")
print("\nAll toolkit-created resources tagged and verified.")

---
## Step 9: Test the Agent

Each test exercises different capabilities of the production agent:

| Test | What it exercises |
|------|-------------------|
| 1. Search hotels in Lisbon | Semantic routing → `search_available_hotels` Lambda → DynamoDB scan |
| 2. Book for 15 guests | Steering rule: max 10 guests → agent self-corrects to 10 |
| 3. Full booking flow | Multi-turn session: search → book → pay → confirm (same session ID) |
| 4. Sold out hotel | AnyCompany Rome Centro has 0 rooms → agent handles gracefully |
| 5. Budget search | Cross-country search under $100 → finds cheapest hotels |
| 6. Confirm without payment | Hard guardrail (BookingGuardrailsHook) → BLOCKED |
| 7. Same-day booking | Steering rule: advance booking required → agent adjusts to tomorrow |

In [ ]:
import uuid
import json

def invoke_agent(prompt, session_id=None, show_tools=True):
    """
    Invoke the deployed agent and display tool usage from Strands Agent.
    
    Args:
        prompt (str): The user's question or request
        session_id (str, optional): UUID for the session
        show_tools (bool): Display which tools the agent called
    
    Returns:
        tuple: (response_text, session_id, tools_called)
    """
    if not session_id:
        session_id = str(uuid.uuid4())
    
    print(f"\n{'='*70}")
    print(f"💬 User: {prompt}")
    print(f"🔑 Session: {session_id[:8]}...")
    print(f"{'='*70}")
    
    try:
        # Invoke agent via boto3
        payload = json.dumps({"prompt": prompt}).encode()
        response = agentcore_data.invoke_agent_runtime(
            agentRuntimeArn=RUNTIME_ARN,
            runtimeSessionId=session_id,
            payload=payload,
            qualifier="DEFAULT"
        )
        
        # Read the response
        response_body = response['response'].read()
        content = response_body.decode('utf-8')
        
        # Try to parse as JSON (new agent version)
        try:
            result = json.loads(content)
            if isinstance(result, dict):
                # New version - JSON with tools_used
                response_text = result.get('response', str(result))
                tools_called = result.get('tools_used', [])
            else:
                # Old version - plain string
                response_text = content
                tools_called = []
        except json.JSONDecodeError:
            # Old version - plain string
            response_text = content
            tools_called = []
        
        # Display tools if any were used
        if show_tools and tools_called:
            print(f"\n🔧 Tools called:")
            for i, tool in enumerate(tools_called, 1):
                print(f"   {i}. {tool}")
            print()
        
        print(f"🤖 Agent:\n{response_text}\n")
        
        return response_text, session_id, tools_called
    
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None, session_id, []

In [ ]:
# Test 5b: Neo4j Knowledge Graph Query — ONLY answerable via graph
# DynamoDB has 18 "AnyCompany" hotels with pricing/availability
# Neo4j has 300 global hotels with amenities, ratings, policies
# This query asks about amenities — data that exists ONLY in Neo4j
_ = invoke_agent("Which hotels have a swimming pool and spa amenities?")

In [ ]:
# Test 1: Search for hotels — exercises semantic routing + DynamoDB scan
_ = invoke_agent("Search for hotels in Lisbon")

In [ ]:
# Test 2: Steering rule, max 10 guests.
# The agent should self-correct by splitting into 2 rooms (10 + 5) instead of
# blocking.
from datetime import datetime, timedelta

check_in = (datetime.now() + timedelta(days=30)).strftime("%Y-%m-%d")
check_out = (datetime.now() + timedelta(days=34)).strftime("%Y-%m-%d")
_ = invoke_agent(
    f"Book AnyCompany Paris Central for 15 guests from {check_in} to {check_out}"
)

In [ ]:
# Test 3: Full booking flow in a single session
# search → validate → book → pay → confirm
session_id = str(uuid.uuid4())

# Step 1: Search
_ = invoke_agent("Find me a 4-star hotel in London", session_id=session_id)

In [ ]:
# Test 3 continued: Full booking flow - provide all details in each call
session_id = str(uuid.uuid4())

# Step 1: Search
_ = invoke_agent("Find me a 4-star hotel in London", session_id=session_id)

In [ ]:
# Step 2: Book, must provide hotel name explicitly (no memory).
from datetime import datetime, timedelta

check_in = (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d")
check_out = (datetime.now() + timedelta(days=11)).strftime("%Y-%m-%d")
book_response, _, _ = invoke_agent(
    f"Book AnyCompany London City Hotel for 2 guests from {check_in} to {check_out}",
    session_id=session_id,
)

In [ ]:
# Step 3: Pay and confirm, extract the booking ID from Step 2's response above.
# Since there is no memory, we pull the booking ID out of the previous response
# text instead of relying on the agent to remember it.
import re

match = re.search(r"BK-[A-Z0-9]{8}", book_response or "")
booking_id = match.group(0) if match else ""
_ = invoke_agent(
    f"Process payment for booking ID {booking_id} and confirm it",
    session_id=session_id,
)

In [ ]:
# Test 5: Budget search across countries
_ = invoke_agent("Find hotels under $100 per night")

In [ ]:
# Test 6: Hard guardrail — confirm without payment
# The BookingGuardrailsHook BLOCKS this at the framework level
_ = invoke_agent("Confirm booking BK-DOESNOTEXIST")

---
## Deployment Complete

You have successfully deployed a production-ready hotel booking agent with:

✅ **DynamoDB tables** — Hotels, Bookings, SteeringRules  
✅ **Lambda functions** — 7 booking tools + 1 Neo4j query  
✅ **AgentCore Gateway** — MCP semantic routing  
✅ **AgentCore Runtime** — Strands agent with Bedrock Claude

**What's next:**

- **Module 7** — Add long-term memory (remembers across sessions)
- **Module 8** — Cleanup (delete all workshop resources)

Module 7 reuses the Gateway, Lambdas, and IAM roles created here.

In [ ]:
# Display deployment variables

print("✅ Module 6 Deployment Complete!")
print()
print("Deployment Summary:")
print("=" * 60)
print(f"Region:              {REGION}")
print(f"Gateway ID:          {GATEWAY_ID}")
print(f"Gateway URL:         {GATEWAY_URL}")
print(f"Runtime ARN:         {RUNTIME_ARN}")
print(f"AgentCore Role:      {AGENTCORE_ROLE_ARN}")
print(f"DynamoDB Tables:     {HOTELS_TABLE}, {BOOKINGS_TABLE}, {STEERING_RULES_TABLE}")
print(f"Lambda Functions:    {len(LAMBDA_ARNS)} deployed")
print("=" * 60)
print()
print("Next: Run Module 7 to add long-term memory capabilities")
print("      Run Module 8 to clean up all resources")

---
## Save Variables for Module 7

Module 7 (Memory) reuses these resources. The variables below are already defined in this notebook session — Module 7 will recover them automatically by querying AWS APIs.

In [ ]:
# Verify all required variables are set for Module 7
print("Variables available for Module 7:")
print("=" * 60)

required_vars = {
    "REGION": REGION,
    "BOOKINGS_TABLE": BOOKINGS_TABLE,
    "AGENTCORE_ROLE_ARN": AGENTCORE_ROLE_ARN,
    "GATEWAY_ID": GATEWAY_ID,
    "GATEWAY_URL": GATEWAY_URL,
    "RUNTIME_ARN": RUNTIME_ARN,
}

all_set = True
for var_name, var_value in required_vars.items():
    status = "✅" if var_value else "❌ NOT SET"
    print(f"  {var_name:25s} = {var_value or 'MISSING'} {status if not var_value else ''}")
    if not var_value:
        all_set = False

print("=" * 60)

if all_set:
    print("\n✅ All variables ready for Module 7")
    print("\nModule 7 will recover these automatically using:")
    print("  - IAM role: iam.get_role(RoleName='workshop-AgentCoreExecutionRole')")
    print("  - Gateway: agentcore.list_gateways() + get_gateway()")
else:
    print("\n⚠️  Some variables missing. Re-run the deployment cells.")